In [2]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings


In [3]:
# Ignore warnings
warnings.filterwarnings("ignore")


In [4]:
# Load the data
df = pd.read_csv(r"C:\Users\enriq\Desktop\Master\Data Science\Clases Juan\Día 5 (Intro ML)\data\teratogenicity_data (1).txt")
df = df.set_index("ID")
df.head()

,Teratogenicity,LogP,Aromatic_rings,Molecular_weight,H_bond_acceptors,H_bond_donors,Polar_surface_area,Rotatable_bonds,C,F,...,SPST,TCCX,TCOT,TXCX,XCCX,double_bonds,triple_bonds,number_of_charges,net_charge,number_of_rings
ID,,,,,,,,,,,,,,,,,,,,,
CHEMBL1466,1,2.79,2,336.29,6,2,93.06,2,19,0,...,0,0,0,0,0,4,0,0,0,4
CHEMBL633,1,7.24,3,645.31,3,0,42.68,11,25,0,...,0,0,0,0,0,1,0,0,0,3
CHEMBL1201039,1,1.97,2,431.94,6,2,160.75,5,15,0,...,0,0,0,0,0,5,0,0,0,3
CHEMBL170365,-1,-0.18,0,44.05,1,0,17.07,0,2,0,...,0,0,0,0,0,1,0,0,0,0
CHEMBL223520,1,-7.14,0,484.50,15,11,282.61,6,18,0,...,0,0,0,0,0,0,0,0,0,3


In [5]:
# Obtain X and y matrices
X = df.drop("Teratogenicity", axis = 1).values
X = sm.add_constant(X)
Y = df.Teratogenicity.values
Y[Y==-1]=0


# Filter
Filtrar los valors de X que tengan poca varianza. Se puede utilizar la funcion VarianceThreshold de sklearn.feature_selection.
Y transformar los datos utilizando la función StandardScaler de sklearn.preprocessing

In [6]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

# Remove features with low variance
selector = VarianceThreshold(threshold=0.01)
X = selector.fit_transform(X)

# Standardize features by removing the mean and scaling to unit variance
scaler = StandardScaler()
X = scaler.fit_transform(X)
print(X.shape)


(521, 212)


# Split in train and test
Se puede utilizar train_test_split de la libreria sklearn.model_selection.
Queremos 50% de datos para train y para test.

In [7]:

from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.5, random_state=42)

# Print the shapes to verify the split
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


Training set shape: (260, 212)
Test set shape: (261, 212)


# Contruir modelos

Se pide:
- Modelo de regresión logística utilizando lasso
- Modelo de regresión logística utilizando elastic-net

Para cada modelo calcular el AUROC y al accuracy y el número de varialbes utilizadas en el modelo.

In [8]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

In [9]:
# 1. Logistic Regression with LASSO
lasso_model = LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear', random_state=42)
lasso_model.fit(X_train, y_train)

# Predictions and metrics for LASSO
lasso_pred = lasso_model.predict(X_test)
lasso_pred_proba = lasso_model.predict_proba(X_test)[:, 1]
lasso_auroc = roc_auc_score(y_test, lasso_pred_proba)
lasso_accuracy = accuracy_score(y_test, lasso_pred)
lasso_n_features = np.sum(lasso_model.coef_ != 0)

print("LASSO Results:")
print(f"AUROC: {lasso_auroc:.3f}")
print(f"Accuracy: {lasso_accuracy:.3f}")
print(f"Number of features used: {lasso_n_features}")
print("-" * 50)

# 2. Logistic Regression with Elastic Net
elastic_model = LogisticRegressionCV(cv=5, penalty='elasticnet', solver='saga', 
                                   l1_ratios=[.1, .5, .7, .9, .95, .99, 1],
                                   random_state=42)
elastic_model.fit(X_train, y_train)

# Predictions and metrics for Elastic Net
elastic_pred = elastic_model.predict(X_test)
elastic_pred_proba = elastic_model.predict_proba(X_test)[:, 1]
elastic_auroc = roc_auc_score(y_test, elastic_pred_proba)
elastic_accuracy = accuracy_score(y_test, elastic_pred)
elastic_n_features = np.sum(elastic_model.coef_ != 0)

print("Elastic Net Results:")
print(f"AUROC: {elastic_auroc:.3f}")
print(f"Accuracy: {elastic_accuracy:.3f}")
print(f"Number of features used: {elastic_n_features}")


LASSO Results:
AUROC: 0.688
Accuracy: 0.670
Number of features used: 119
--------------------------------------------------
Elastic Net Results:
AUROC: 0.671
Accuracy: 0.644
Number of features used: 115


# Construir Modelo Seleccionando mejores k features

- Utilizar la función SelectKBest de sklearn.feature_selection para seleccionar las mejores k features. 
- Con esas k variables realizar la regresión logística con cross-validaton con cv = 5, optimizando los valores de roc_auc.
- Calcular la media de los 5 aurocs obtenidos.
- Seleccionar la k con el valor más alto de media de aurocs y con este valor de k entrenar todo el train y evaluar el modelo con el test.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.model_selection import cross_val_score


In [11]:
# Try different values of k features
k_values = range(1, X_train.shape[1] + 1)
mean_aurocs = []

# For each k value
for k in k_values:
    # Select k best features
    selector = SelectKBest(score_func=f_classif, k=k)
    X_k = selector.fit_transform(X_train, y_train)
    
    # Create and evaluate model with cross-validation
    model = LogisticRegression(random_state=42)
    aurocs = cross_val_score(model, X_k, y_train, cv=5, scoring='roc_auc')
    
    # Store mean AUROC
    mean_aurocs.append(np.mean(aurocs))

# Find best k
best_k = k_values[np.argmax(mean_aurocs)]
print(f"Best k: {best_k} with mean AUROC: {max(mean_aurocs):.3f}")

# Train final model with best k
final_selector = SelectKBest(score_func=f_classif, k=best_k)
X_train_selected = final_selector.fit_transform(X_train, y_train)
X_test_selected = final_selector.transform(X_test)

final_model = LogisticRegression(random_state=42)
final_model.fit(X_train_selected, y_train)

# Evaluate on test set
y_pred_proba = final_model.predict_proba(X_test_selected)[:, 1]
final_auroc = roc_auc_score(y_test, y_pred_proba)
final_accuracy = accuracy_score(y_test, final_model.predict(X_test_selected))

print("\nFinal Model Results:")
print(f"Test AUROC: {final_auroc:.3f}")
print(f"Test Accuracy: {final_accuracy:.3f}")
print(f"Number of features used: {best_k}")


Best k: 53 with mean AUROC: 0.741

Final Model Results:
Test AUROC: 0.665
Test Accuracy: 0.644
Number of features used: 53
